# GeoAI Digital Asset Pipeline - Getting Started

This notebook demonstrates the basic capabilities of the GeoAI Digital Asset Pipeline for geospatial analysis with AI/ML integration.

## Table of Contents

1. [Setup and Configuration](#Setup-and-Configuration)
2. [Data Loading and Exploration](#Data-Loading-and-Exploration)
3. [Feature Extraction](#Feature-Extraction)
4. [Machine Learning Classification](#Machine-Learning-Classification)
5. [Quality Assurance](#Quality-Assurance)
6. [Export and Visualization](#Export-and-Visualization)

## Setup and Configuration

First, let's import the necessary modules and set up the configuration.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

# Import pipeline modules
from utils.config import Config
from workflow.logger import PipelineLogger, setup_logging
from workflow.progress_tracker import ProgressTracker

# Setup logging
setup_logging(level='INFO')

# Load configuration
config = Config.default()
logger = PipelineLogger.get_logger('Notebook')

print(f"GeoAI Pipeline v{config.project.version}")
print(f"Project: {config.project.name}")

## Data Loading and Exploration

In this section, we'll create sample data for demonstration purposes. In production, you would load actual satellite imagery or vector data.

In [ ]:
# Create synthetic satellite imagery data for demonstration
np.random.seed(42)

# Simulate a 100x100 pixel image with 6 spectral bands
image_size = (100, 100)

# Create bands with different characteristics
bands = {
    'blue': np.random.normal(500, 100, image_size),
    'green': np.random.normal(800, 150, image_size),
    'red': np.random.normal(1000, 200, image_size),
    'nir': np.random.normal(2000, 400, image_size),
    'swir1': np.random.normal(1500, 300, image_size),
    'swir2': np.random.normal(1000, 200, image_size),
}

# Ensure positive values
for band in bands:
    bands[band] = np.clip(bands[band], 1, None)

print(f"Created synthetic image: {image_size[0]}x{image_size[1]} pixels")
print(f"Bands: {list(bands.keys())}")

## Feature Extraction

Extract spectral indices and other features from the imagery.

In [ ]:
from pipeline.feature_extractor import FeatureExtractor

# Initialize feature extractor
extractor = FeatureExtractor(config=config, logger=logger)

# Calculate spectral indices
indices = extractor.calculate_spectral_indices(bands)

print("Calculated Spectral Indices:")
for name, data in indices.items():
    print(f"  {name}: min={data.min():.3f}, max={data.max():.3f}, mean={data.mean():.3f}")

In [ ]:
# Visualize NDVI
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(bands['nir'], cmap='gray')
plt.title('NIR Band')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(bands['red'], cmap='gray')
plt.title('Red Band')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(indices['NDVI'], cmap='RdYlGn', vmin=-1, vmax=1)
plt.title('NDVI')
plt.colorbar(label='NDVI value')
plt.axis('off')

plt.tight_layout()
plt.show()

## Machine Learning Classification

Train a classifier to categorize pixels into land cover classes.

In [ ]:
from ml.classifier import SpatialClassifier

# Prepare feature matrix
feature_matrix = np.stack([
    bands['red'].flatten(),
    bands['green'].flatten(),
    bands['blue'].flatten(),
    bands['nir'].flatten(),
    indices['NDVI'].flatten(),
    indices['NDWI'].flatten(),
    indices['NDBI'].flatten(),
], axis=1)

# Create synthetic training labels (for demonstration)
# In production, you would use actual labeled data
ndvi_flat = indices['NDVI'].flatten()
labels = np.zeros_like(ndvi_flat, dtype=int)
labels[ndvi_flat > 0.4] = 1  # Vegetation
labels[ndvi_flat < 0.1] = 2  # Water/Bare soil

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Class distribution: {np.bincount(labels)}")

In [ ]:
# Initialize and train classifier
classifier = SpatialClassifier(
    algorithm='random_forest',
    n_estimators=50,
    max_depth=10,
    config=config,
    logger=logger
)

# Train the model
feature_names = ['red', 'green', 'blue', 'nir', 'ndvi', 'ndwi', 'ndbi']
class_names = ['bare_soil', 'vegetation', 'water']

training_result = classifier.train(
    feature_matrix,
    labels,
    feature_names=feature_names,
    class_names=class_names,
    validation_size=0.2
)

print(f"\nTraining Results:")
print(f"  Training Accuracy: {training_result.training_accuracy:.3f}")
print(f"  Validation Accuracy: {training_result.validation_accuracy:.3f}")
print(f"  CV Mean Score: {training_result.metrics.get('cv_mean', 0):.3f}")

In [ ]:
# Make predictions
predictions = classifier.predict(feature_matrix)
prediction_map = predictions.reshape(image_size)

# Visualize predictions
plt.figure(figsize=(8, 6))
plt.imshow(prediction_map, cmap='tab10', vmin=0, vmax=2)
plt.title('Land Cover Classification')
plt.colorbar(ticks=[0, 1, 2], label='Class')
plt.axis('off')
plt.tight_layout()
plt.show()

## Quality Assurance

Run quality checks on the classification results.

In [ ]:
from pipeline.quality_assurance import QualityAssurance
import geopandas as gpd
from shapely.geometry import Point

# Create sample vector data for QA demonstration
sample_points = [
    Point(i, j) for i in range(10) for j in range(10)
]
sample_values = np.random.rand(100)

gdf = gpd.GeoDataFrame({
    'id': range(100),
    'value': sample_values,
    'category': np.random.choice(['A', 'B', 'C'], 100),
    'geometry': sample_points
})

# Run quality checks
qa = QualityAssurance(config=config, logger=logger)
report = qa.run_all_checks(gdf)

print(f"Quality Assurance Report")
print(f"="*40)
print(f"Total Features: {report.total_features}")
print(f"Checks Run: {report.checks_run}")
print(f"Checks Passed: {report.checks_passed}")
print(f"Overall Score: {report.overall_score:.1f}%")

In [ ]:
# Display detailed results
print("\nDetailed Check Results:")
print("-" * 40)
for result in report.results:
    status = "✓" if result.passed else "✗"
    print(f"{status} {result.check_name}: {result.message}")

## Export and Visualization

Export results and create summary visualizations.

In [ ]:
from pipeline.asset_manager import AssetManager, AssetType

# Create asset manager
manager = AssetManager(config=config, logger=logger, asset_dir='./outputs')

# Create asset from classification results
asset = manager.create_asset(
    data=prediction_map,
    name='land_cover_classification',
    asset_type=AssetType.RASTER,
    metadata={
        'description': 'Land cover classification results',
        'classes': class_names,
        'accuracy': training_result.validation_accuracy
    }
)

print(f"Created asset: {asset.id}")
print(f"Asset type: {asset.asset_type.value}")
print(f"Status: {asset.status.value}")

In [ ]:
# Calculate and display class statistics
unique, counts = np.unique(prediction_map, return_counts=True)
total_pixels = prediction_map.size

print("\nClassification Statistics:")
print("-" * 40)
for class_id, count in zip(unique, counts):
    class_name = class_names[class_id] if class_id < len(class_names) else f'Class {class_id}'
    percentage = (count / total_pixels) * 100
    print(f"{class_name}: {count} pixels ({percentage:.1f}%)")

## Summary

This notebook demonstrated the basic workflow of the GeoAI Digital Asset Pipeline:

1. **Configuration**: Set up the pipeline configuration
2. **Feature Extraction**: Calculate spectral indices from multispectral imagery
3. **Classification**: Train and apply a machine learning classifier
4. **Quality Assurance**: Run quality checks on the results
5. **Asset Management**: Export and manage the results as digital assets

For more advanced use cases, see the other notebooks in this directory:
- `02_building_extraction.ipynb` - Object detection for buildings
- `03_change_detection.ipynb` - Multi-temporal change analysis
- `04_spatial_clustering.ipynb` - Spatial pattern analysis